In [1]:
!pip install -qq bitsandbytes peft trl math_verify

# GRPO


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_type=torch.bfloat16
)


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## Dataset for GRPO

In [3]:
dataset_id = "AI-MO/NuminaMath-TIR"
train_dataset, test_dataset = load_dataset(dataset_id, split=["train[:5%]", "test[:5%]"])

In [4]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning "
    "process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> answer here </answer>"
)


def make_conversation(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["problem"]},
        ],
    }

columns_to_remove = ['messages', 'problem']
train_dataset = train_dataset.map(make_conversation, remove_columns=columns_to_remove)
test_dataset = test_dataset.map(make_conversation, remove_columns=columns_to_remove)

In [7]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.01,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)

In [8]:
import re


def format_reward(completions, **kwargs):
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"
    completion_contents = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, content) for content in completion_contents]
    rewards_list = [1.0 if match else 0.0 for match in matches]
    return [1.0 if match else 0.0 for match in matches]

In [9]:
from math_verify import parse, LatexExtractionConfig, verify



def accuracy_reward(completions, **kwargs):
    """Reward function that checks if the completion is the same as the ground truth."""
    solutions = kwargs["solution"]
    completion_contents = [completion[0]["content"] for completion in completions]
    rewards = []
    for content, solution in zip(completion_contents, solutions):
        gold_parsed = parse(solution, extraction_mode="first_match", extraction_config=[LatexExtractionConfig()])
        answer_parsed = parse(content, extraction_mode="first_match", extraction_config=[LatexExtractionConfig()])
        if len(gold_parsed) != 0:
            try:
                rewards.append(float(verify(answer_parsed, gold_parsed)))
            except Exception:
                rewards.append(0.0)
        else:
            rewards.append(1.0)
    return rewards

In [10]:
from trl import GRPOTrainer, GRPOConfig


training_args_stable = GRPOConfig(
    output_dir="Qwen/Qwen2.5-1.5B-Instruct-math-v0145",
    learning_rate=1e-5,
    temperature=0.6,
    remove_unused_columns=False,
    gradient_accumulation_steps=16,
    num_train_epochs=2,
    bf16=True,
    max_completion_length=128,
    num_generations=4,
    report_to="none",
    logging_steps=10,
    push_to_hub=False,
    save_strategy="steps",
    save_steps=10,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward, accuracy_reward],
    args=training_args_stable,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

In [11]:
trainer.train()

Step,Training Loss
10,0.047621
20,0.052970
30,0.057846
40,0.059908
50,0.075380
60,0.083960
70,0.088570
80,0.094575
90,0.105970
100,0.119269


TrainOutput(global_step=226, training_loss=0.10747776015669899, metrics={'train_runtime': 2612.2981, 'train_samples_per_second': 2.773, 'train_steps_per_second': 0.087, 'total_flos': 0.0, 'train_loss': 0.10747776015669899, 'num_tokens': 7711224.0, 'completions/mean_length': 87.65494791666667, 'completions/min_length': 19.666666666666668, 'completions/max_length': 128.0, 'completions/clipped_ratio': 0.3463541666666667, 'completions/mean_terminated_length': 66.35390726725261, 'completions/min_terminated_length': 19.666666666666668, 'completions/max_terminated_length': 125.33333333333333, 'rewards/format_reward/mean': 0.640625, 'rewards/format_reward/std': 0.480065181851387, 'rewards/accuracy_reward/mean': 0.053385416666666664, 'rewards/accuracy_reward/std': 0.21943117926518121, 'reward': 0.6940104166666666, 'reward_std': 0.5525113344192505, 'frac_reward_zero_std': 0.3177083333333333, 'entropy': 0.2823574763412277, 'clip_ratio/low_mean': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/regio

In [13]:
trainer.save_model("Qwen/Qwen2.5-1.5B-Instruct-math-v0145")


In [14]:
trainer.push_to_hub("Qwen/Qwen2.5-1.5B-Instruct-math-v0145")

CommitInfo(commit_url='https://huggingface.co/nickkats1/Qwen2.5-1.5B-Instruct-math-v0145/commit/1549b7c67845669898fe56d36331fb0458ea1788', commit_message='Qwen/Qwen2.5-1.5B-Instruct-math-v0145', commit_description='', oid='1549b7c67845669898fe56d36331fb0458ea1788', pr_url=None, repo_url=RepoUrl('https://huggingface.co/nickkats1/Qwen2.5-1.5B-Instruct-math-v0145', endpoint='https://huggingface.co', repo_type='model', repo_id='nickkats1/Qwen2.5-1.5B-Instruct-math-v0145'), pr_revision=None, pr_num=None)

In [15]:
del trainer
torch.cuda.empty_cache()
import gc
gc.collect()

15843

In [20]:
!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 69.5 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [21]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct-math-v0145"

tokenizer = AutoTokenizer.from_pretrained(model_name)


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

W1121 13:00:26.648000 1862 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

In [22]:
import time


def generate_with_reasoning(prompt):
    prompt = " ".join(entry["content"] for entry in prompt)


    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)


    start_time = time.time()
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_length=500)
    end_time = time.time()


    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)


    inference_duration = end_time - start_time


    num_input_tokens = inputs["input_ids"].shape[1]
    num_generated_tokens = output_ids.shape[1] - num_input_tokens

    return generated_text, inference_duration, num_generated_tokens

In [30]:
prompt = test_dataset["prompt"][0]
generated_text, inference_duration, num_generated_tokens = generate_with_reasoning(prompt)
print(generated_text)

A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think><answer> answer here </answer> In 1988, a person's age was equal to the sum of the digits of their birth year. How old was this person? To find out how old the person was in 1988, we need to determine what year they were born.
<think> Let's start by assuming that the person was born in the early 20th century (around 1900s). We can check if any such years exist where the person's age in 1988 is equal to the sum of the digits of their birth year.</think>
<answer> 1905 </answer> 
In 1905, the person would be 79 years old because 1 + 9 + 0 + 5 = 15, which is not correct. Let's try another year.
<think> Let's assume the person was